# STAT 764 · Capstone Project 

Jake

Caleb


Cole

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [2]:
import numpy as np
import pandas as pd

# Load the TidyTuesday hotels dataset directly into a Pandas DataFrame
hotels = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv')

# Verify it loaded correctly
print(hotels.head())
print("Shape:", hotels.shape)

          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0                     0       2  ...    No Deposit   
1     

In [3]:
# Cancellation rate across all rows (dataset column is spelled is_canceled).
canceled = hotels["is_canceled"].eq(1)
print(f"Cancellation rate: {canceled.mean():.2%} ({canceled.sum():,}/{len(hotels):,} rows)")

# Count missing values for every variable, including columns with zero missing values.
print("\nMissing values by variable:")
print(hotels.isna().sum().to_string())

Cancellation rate: 37.04% (44,224/119,390 rows)

Missing values by variable:
hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type         

In [4]:
# Libraries
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

In [5]:
# Count exact row patterns, from most to least frequent
row_counts = (
    hotels.groupby(hotels.columns.tolist(), dropna=False, sort=False)
    .size()
    .sort_values(ascending=False)
)

print("Most repeated exact rows:")
print(row_counts.head(10).to_string())
print(f"\nRows beyond the first copy: {hotels.duplicated().sum():,}")

Most repeated exact rows:
hotel       is_canceled  lead_time  arrival_date_year  arrival_date_month  arrival_date_week_number  arrival_date_day_of_month  stays_in_weekend_nights  stays_in_week_nights  adults  children  babies  meal  country  market_segment  distribution_channel  is_repeated_guest  previous_cancellations  previous_bookings_not_canceled  reserved_room_type  assigned_room_type  booking_changes  deposit_type  agent  company  days_in_waiting_list  customer_type  adr    required_car_parking_spaces  total_of_special_requests  reservation_status  reservation_status_date
City Hotel  1            277        2016               November            46                        7                          1                        2                     2       0.0       0       BB    PRT      Groups          TA/TO                 0                  0                       0                               A                   A                   0                Non Refund    NaN    NaN    

# Removing Duplicated Rows

In [6]:
print(f"Rows belonging to duplicate groups: {hotels.duplicated(keep=False).sum():,}")
before = len(hotels)
hotels = hotels.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(hotels):,} duplicate rows; {len(hotels):,} rows remain.")

Rows belonging to duplicate groups: 40,165
Removed 31,994 duplicate rows; 87,396 rows remain.


# Deposit Leaks

In [7]:
# Remove deposit_type because it can leak information about cancellation.
hotels = hotels.drop(columns=["deposit_type"])

In [8]:
# Cancellation rate across all rows (dataset column is spelled is_canceled).
canceled = hotels["is_canceled"].eq(1)
print(f"Cancellation rate: {canceled.mean():.2%} ({canceled.sum():,}/{len(hotels):,} rows)")

# Count missing values for every variable, including columns with zero missing values.
print("\nMissing values by variable:")
print(hotels.isna().sum().to_string())

Cancellation rate: 27.49% (24,025/87,396 rows)

Missing values by variable:
hotel                                 0
is_canceled                           0
lead_time                             0
arrival_date_year                     0
arrival_date_month                    0
arrival_date_week_number              0
arrival_date_day_of_month             0
stays_in_weekend_nights               0
stays_in_week_nights                  0
adults                                0
children                              4
babies                                0
meal                                  0
country                             452
market_segment                        0
distribution_channel                  0
is_repeated_guest                     0
previous_cancellations                0
previous_bookings_not_canceled        0
reserved_room_type                    0
assigned_room_type                    0
booking_changes                       0
agent                             12193
comp

In [9]:
arrival_date = pd.to_datetime(
    hotels["arrival_date_year"].astype(str)
    + "-"
    + hotels["arrival_date_month"]
    + "-"
    + hotels["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
)

edges = pd.date_range(arrival_date.min(), arrival_date.max(), periods=11)
fold_id = pd.cut(arrival_date, bins=edges, labels=False, include_lowest=True)

# Ten DataFrames, ordered from earliest to latest arrival period
time_folds = [
    hotels.loc[fold_id.eq(i)].copy()
    for i in range(10)
]

for i, fold in enumerate(time_folds, start=1):
    dates = arrival_date.loc[fold.index]
    print(f"Fold {i}: {dates.min().date()} to {dates.max().date()} "
          f"({len(fold):,} rows)")

Fold 1: 2015-07-01 to 2015-09-18 (5,875 rows)
Fold 2: 2015-09-19 to 2015-12-06 (5,950 rows)
Fold 3: 2015-12-07 to 2016-02-23 (5,480 rows)
Fold 4: 2016-02-24 to 2016-05-12 (9,703 rows)
Fold 5: 2016-05-13 to 2016-07-31 (9,709 rows)
Fold 6: 2016-08-01 to 2016-10-18 (10,607 rows)
Fold 7: 2016-10-19 to 2017-01-05 (8,917 rows)
Fold 8: 2017-01-06 to 2017-03-25 (8,544 rows)
Fold 9: 2017-03-26 to 2017-06-12 (11,235 rows)
Fold 10: 2017-06-13 to 2017-08-31 (11,376 rows)


A single fill looks as sure as a measured value, but one county's draws spread over a point or more (Autauga
County: 1.65 to 5.05). Averages over many counties barely move (every draw gives about 3.6); one county's number
does. Fitting the
model once on each of several draws, and reporting how much the answer moves, is **multiple imputation**, the usual
way to carry that uncertainty forward.

**Before filling any blank, ask:**

1. **Why is it blank?** Does "blank" mean "none" or "not applicable"? Was it unknown at the moment of prediction?
   MCAR, MAR or MNAR? Who would know?
2. **Does the blank say something?** Compare the outcome when blank and when filled. If it differs, keep the blank
   as information: a category, or an indicator column.
3. **Who does dropping remove?** If you drop rows, say who they are, because the model will not be about them.

## Studio · the capstone data clinic

**Run these checks on your own team's capstone data.** Work with your team and compare as you go, but each of you
uploads your own notebook. If your team's data will not load on your laptop today, use the readmission data above
(`df = enc`, `OUTCOME = "came_back"`) and say so in your answers.

| | what you practice | about |
|---|---|---|
| **Question 1** | counting the blanks, including the ones in disguise | 10 minutes |
| **Question 2** | asking whether a blank says something about the outcome | 15 minutes |
| **Question 3** | a missing-data plan for your capstone | 15 minutes |

**All three questions count.** If you do not finish in class, finish at home: the notebook is due 24 hours after
class.

**How to work.**

1. Run the notebook from the top (**Run All**) first: it takes under a minute.
2. In the first studio cell, load the table your team models from (copy the loading code from your capstone
   notebook) into `df`, and set `OUTCOME` to the name of your 0/1 outcome column.
3. Before questions 1 and 2, **write your guess** as a comment in the cell, and keep it when the answer disagrees.
   The Compare uses it.
4. Each question ends with sentences to answer. Write them in a comment or a markdown cell under its output.
5. Save and upload to Canvas under "Studio notebook — Day 22" by **Friday, November 20, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `FileNotFoundError` | The path in your loading code points at your capstone folder; copy it from your capstone notebook, or use the full path. |
| `KeyError` for your outcome | `OUTCOME` must match a column name exactly: `print(df.columns.tolist())`. |
| Your outcome is not 0/1 | Make it one first, as your capstone notebook does, e.g. `df["late"] = (df["ArrDelay"] >= 15).astype(int)` (use your file's column name). Careful: that turns a **blank** delay (a cancelled flight) into 0, on time. Count those blanks first. |
| Your outcome itself has blanks | That is a finding: say which rows and why (a cancelled flight has no arrival delay). Use `df.dropna(subset=[OUTCOME])` for question 2, and count what you dropped. |
| No column has a blank | Look for disguised ones (question 1b). If your team already dropped rows, run questions 1 and 2 on the file **before** those rows were dropped. |
| Question 2 prints one-row tables | Those columns are blank in every row (some flight files have dozens); use only the partly blank ones. |

### 1. Count the blanks

- **(a)** For every column with a blank: how many, and what share of rows. Largest share first:
  `df.isna().sum()`, then `df.isna().mean()`, keeping the columns above 0.
- **(b)** Blanks in disguise. Count, in each text column, the values that mean "blank" without being one:
  `df.select_dtypes(exclude="number").isin(["?", "Undefined", "Unknown", "unknown", "-"]).sum()`. And look
  for impossible numbers your data might use for "unknown" (a 0 where 0 cannot happen, a 999). Note: when pandas read
  the file it already turned the words NULL, NA and N/A, and empty cells, into blanks, so some of (a)'s blanks were
  words in the file. Ask what they meant: the hotel file's NULL in `agent` means "no agent".
- **(c)** If your team has already dropped rows (duplicates, cancelled flights, bookings with no guests), say how
  many and why.

Guess first: which column of your data has the most blanks?

**Answer in two or three sentences:** which columns have blanks, and for each of the biggest, what does "blank"
mean there? Is it really missing, or does it mean "none" or "does not apply"?

### 2. Does a blank say something?

- **(a)** For the three **partly** blank columns (blank in some rows, not all) with the most blanks (fewer if your
  data has fewer), the outcome rate when the column is blank and when it is filled, with how many rows each. One
  table per column is fine:

  ```python
  share = df.isna().mean()
  for c in share[(share > 0) & (share < 1)].nlargest(3).index:
      print(df.groupby(df[c].isna())[OUTCOME].agg(["size", "mean"]), "\n")
  ```
- **(b)** For the column whose blank differs most, compare blank and filled rows on one other column that might
  explain it (as Section 1 compared tested and untested stays by age).

Guess first: will any blank change the outcome rate by more than 5 points?

**Answer in two or three sentences:** which blank says the most about the outcome, and by how much? From (b), are
the blanks random? If not, what reason the data do not record could make them MNAR, and who would know?

### 3. Your team's missing-data plan

Write a markdown table with one row for each of your three columns from question 2:

| column | why it is blank | what we will do | who that affects |
|---|---|---|---|
| | | | |

- **why it is blank:** means "none" · unknown at the moment of prediction · the outcome did not happen as defined ·
  MCAR · MAR · MNAR · censored (known only to be beyond a limit) · don't know yet (and who you would ask).
- **what we will do:** its own category · fill inside the pipeline with an indicator column · drop the column ·
  drop the rows.
- **who that affects:** the rows changed or removed, with a count.

**Answer in two sentences:** which row of your plan are you least sure of, and what would settle it?

In [10]:
# Guess: Company has the most blanks
df = hotels
OUTCOME = "is_canceled"

# (a) Blank counts and shares, largest share first
missing = pd.DataFrame({
    "count": df.isna().sum(),
    "share": df.isna().mean(),
})
missing = missing[missing["count"] > 0].sort_values("share", ascending=False)
print("Missing values:")
print(missing.to_string() if not missing.empty else "No missing values found.")

# (b) Possible blanks encoded as text
text_cols = df.select_dtypes(exclude="number")
text_sentinels = text_cols.isin(["?", "Undefined", "Unknown", "unknown", "-"]).sum()
text_sentinels = text_sentinels[text_sentinels > 0]
print("\nText sentinel counts:")
print(text_sentinels.to_string() if not text_sentinels.empty else "None found.")

# Inspect common numeric sentinel candidates; decide whether each is impossible for its column.
numeric = df.select_dtypes(include="number")
numeric_candidates = numeric.isin([0, -1, 999, 9999]).sum()
numeric_candidates = numeric_candidates[numeric_candidates > 0]
print("\nCounts of numeric candidate values (0, -1, 999, 9999):")
print(numeric_candidates.to_string() if not numeric_candidates.empty else "None found.")

# (c) Cell 10 removed exact duplicate rows; row_counts was computed before that removal.
if "row_counts" in globals():
    removed_duplicates = int((row_counts - 1).clip(lower=0).sum())
    print(f"\nExact duplicate rows removed: {removed_duplicates:,}")
else:
    print("\nExact duplicate rows were removed earlier; the count is not available in this cell.")

Missing values:
          count     share
company   82137  0.939826
agent     12193  0.139514
country     452  0.005172
children      4  0.000046

Text sentinel counts:
meal                    492
market_segment            2
distribution_channel      5

Counts of numeric candidate values (0, -1, 999, 9999):
is_canceled                       63371
lead_time                          5978
stays_in_weekend_nights           35151
stays_in_week_nights               6175
adults                              385
children                          79028
babies                            86482
is_repeated_guest                 83981
previous_cancellations            85711
previous_bookings_not_canceled    83851
booking_changes                   71494
days_in_waiting_list              86536
adr                                1778
required_car_parking_spaces       80083
total_of_special_requests         43894

Exact duplicate rows removed: 31,994


**Answer in two or three sentences:** 
Which columns have blanks, and for each of the biggest, what does "blank" mean there? 
- Country: 452; MCAR
- Agent: 12193; MAR
- Company: 82,137; MAR
- Children: 4; MCAR

Is it really missing, or does it mean "none" or "does not apply"?
- These .isna() rendered values are truely missing values because the function counts NANs and empty cells.

In [11]:
# Guess: At least one blank will change the cancellation rate.
share = df.isna().mean()
partial_cols = share[(share > 0) & (share < 1)].nlargest(3).index

results = {}
for c in partial_cols:
	summary = df.groupby(df[c].isna())[OUTCOME].agg(size="size", rate="mean")
	results[c] = summary
	print(f"{c}: cancellation rate by blank status")
	print(summary, "\n")

# Identify the column with the largest difference in cancellation rates.
gaps = {
	c: abs(summary.loc[True, "rate"] - summary.loc[False, "rate"])
	for c, summary in results.items()
}
most_different = max(gaps, key=gaps.get)
print(f"Largest difference: {most_different} "
	  f"({gaps[most_different]:.1%} points)")

# Compare market segments for rows where that column is blank versus filled.
if "market_segment" in df.columns:
	print(f"\nMarket segment by whether {most_different} is blank (row shares):")
	print(pd.crosstab(
		df[most_different].isna(),
		df["market_segment"],
		normalize="index"
	).round(3))

company: cancellation rate by blank status
          size      rate
company                 
False     5259  0.109146
True     82137  0.285511 

agent: cancellation rate by blank status
        size      rate
agent                 
False  75203  0.298765
True   12193  0.127696 

country: cancellation rate by blank status
          size      rate
country                 
False    86944  0.275936
True       452  0.075221 

Largest difference: country (20.1% points)

Market segment by whether country is blank (row shares):
market_segment  Aviation  Complementary  Corporate  Direct  Groups  \
country                                                              
False              0.003          0.008      0.046   0.134   0.057   
True               0.000          0.009      0.400   0.345   0.011   

market_segment  Offline TA/TO  Online TA  Undefined  
country                                              
False                   0.159      0.593        0.0  
True                    0.071  

**Answer in two or three sentences:** 
Which blank says the most about the outcome, and by how much? 
- Not having a country says the most about our outcome, having a cnacellation rate of .075, 20.1% lower than our average cancellation rate of 27.49%

From (b), are the blanks random? If not, what reason the data do not record could make them MNAR, and who would know?
- Country, choldren seem to be at random
- Agent and company could be MAR depending on other values in our table.

In [12]:
df["country"].value_counts(dropna=False)

country
PRT    27453
GBR    10433
FRA     8837
ESP     7252
DEU     5387
       ...  
NCL        1
KIR        1
SDN        1
ATF        1
SLE        1
Name: count, Length: 178, dtype: int64

## Compare — at the start of next class

We compare answers at the start of next class (Tuesday, November 24), built from the notebooks you upload. So leave
these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 2: your table of outcome rates, blank against filled.
3. Question 3: your plan.

## Exit ticket

> A teammate fills every blank in your data with its column's median, on the whole dataset, before splitting it into
> training and test. In two or three sentences: name two things that could go wrong.

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio notebook — Day 22". Your
two studio points need this answered: a blank exit ticket earns zero. It is not graded for being right — it is how
I find out what landed.**

### Your exit ticket

*1) Leakage*
- When you impute before splitting, you leak information about the test set, into the training set.

*2) Missing At Random
- When the missing values are correlated with other columns, imputing the mean may be worse than imputing via KNN or another technique.